In [ ]:
import numpy as np
from sympy.physics.wigner import wigner_3j
from ksw.Afunctionals import products_3j_array, wigner_J, w3j

In [ ]:
from ducc0.misc import wigner3j_int

In [ ]:
# --- fixed inputs ---
S = 2
n = 0
#Jindex = np.array([0, 0, 0], dtype=int)
Jindex = np.array([-2, 0, 2], dtype=int)

L_list = np.array([0, 1, 2, 3], dtype=int)
#deltaL_list = np.array([-1, 1], dtype=int)
deltaL_list = np.array([-2, -1, 0, 1, 2], dtype=int)


In [ ]:
def w3j_sympy(j1, j2, j3, m1, m2, m3):
    return wigner_3j(int(j1), int(j2), int(j3),
                     int(m1), int(m2), int(m3))

In [ ]:
# --- Sympy ground truth ---
cJ_sympy = np.zeros((len(L_list), len(deltaL_list)), dtype=np.float64)
for iL, L in enumerate(L_list):
    for idL, dL in enumerate(deltaL_list):
        ell = L + dL
        if ell < 0:
            continue

        J_factor = w3j_sympy(S, L, ell, Jindex[0], Jindex[1], Jindex[2])
        cJ_sympy[iL, idL] = float(J_factor.evalf())

Lmax = np.max(L_list)
cJ2_sympy = np.zeros((len(L_list), 2*Lmax+1, len(deltaL_list)), dtype=np.float64)
for idL, dL in enumerate(deltaL_list):
    for iL, L in enumerate(L_list):
        ell = L + dL
        if ell < 0:
            continue

        for M in range(-L, L+1):
            m = - M - n
            if abs(m) > ell:
                continue

            J_factor = w3j_sympy(S, L, ell, n, M, m)
            cJ2_sympy[iL, M + Lmax, idL] = float(J_factor.evalf())


In [ ]:
cJ = wigner_J(S, L_list, deltaL_list, Jindex)
cJ2 = w3j(S, n, L_list, deltaL_list)

In [ ]:
# --- compare ---
abs_err = np.abs(cJ - cJ_sympy)
#print(abs_err)
print(np.max(abs_err))

abs_err2 = np.abs(cJ2-cJ2_sympy)
#print(abs_err2)
print(np.max(abs_err2))

In [ ]:
products = products_3j_array(S, n , L_list, deltaL_list, Jindex)
producs_sympy = cJ2_sympy * cJ_sympy[:, np.newaxis, :]

abs_err_products = np.abs(products - producs_sympy)
#print(abs_err_products)
print(np.max(abs_err_products))

In [ ]:
from ksw.Afunctionals import parity_x
parity = parity_x("B")
print(parity)

In [ ]:
from ksw.Afunctionals import gamma_Z
gammaZ = gamma_Z(x="t", Z="h", L=2, deltaL=2)
print(gammaZ)

In [ ]:
from ksw.estimator import KSW
import numpy as np
from ksw import estimator_core

ksw = KSW.__new__(KSW)
ksw.lmax = 100
ksw.dtype = np.float32
ksw.cdtype = np.complex64
ksw.pol = ["T", "E"]

In [ ]:
thetas_batch = np.array([3.1], dtype=ksw.dtype)
y_m_ell = estimator_core.compute_ylm(thetas_batch, 100, dtype=np.float32)
print(y_m_ell.shape)